# DDIM — deterministic, accelerated diffusion sampling

> Tutorial pair for [`ddim.py`](ddim.py).

## 1. Intuition
A DDPM is slow to sample because it walks back through *every* one of the $T$
noising steps, one network call each. DDIM keeps the same trained noise-predictor
but changes the *sampler*: it defines a family of reverse processes that all share
the DDPM marginals, including a **deterministic** one that can take big jumps.
The result is sharp samples in 20-50 steps instead of 1000, and (at $\eta=0$) a
reproducible map from a fixed noise seed to a fixed sample.

## 2. Concept (the slide)
- **Same training** as DDPM: regress $\epsilon_\theta(x_t,t)$ on the added noise.
- **Different sampling:** a *non-Markovian* reverse process with a free knob
  $\eta\in[0,1]$ controlling injected noise.
- $\eta=1$ recovers the DDPM ancestral sampler; $\eta=0$ is fully deterministic
  (an implicit ODE) and lets you **skip steps** cheaply.
- Pick any decreasing sub-sequence of timesteps $\tau_1>\tau_2>\dots$ to trade
  speed for quality.

## 3. Math derivation — the non-Markovian reverse step

DDIM defines a family of inference processes indexed by $\sigma_t\ge0$ that all
keep the *same* forward marginals $q(x_t\mid x_0)=\mathcal N(\sqrt{\bar\alpha_t}x_0,(1-\bar\alpha_t)I)$,
so a network trained for DDPM is valid unchanged.

**Predict $x_0$.** From $x_t=\sqrt{\bar\alpha_t}x_0+\sqrt{1-\bar\alpha_t}\epsilon$,
$$\hat x_0(x_t,t)=\frac{x_t-\sqrt{1-\bar\alpha_t}\,\epsilon_\theta(x_t,t)}{\sqrt{\bar\alpha_t}}.$$

**Step to an earlier time $s<t$.** The DDIM update is
$$\boxed{\,x_s=\sqrt{\bar\alpha_s}\,\hat x_0
 +\underbrace{\sqrt{1-\bar\alpha_s-\sigma_t^2}\;\epsilon_\theta(x_t,t)}_{\text{direction pointing to }x_t}
 +\sigma_t z\,},\qquad z\sim\mathcal N(0,I).$$
One checks that this preserves $q(x_s\mid x_0)$ for **any** choice of $\sigma_t$.

**The $\eta$ knob.** Set
$$\sigma_t=\eta\sqrt{\frac{1-\bar\alpha_s}{1-\bar\alpha_t}}\sqrt{1-\frac{\bar\alpha_t}{\bar\alpha_s}}.$$
- $\eta=1$ $\Rightarrow$ $\sigma_t$ equals the DDPM posterior std $\tilde\beta_t$
  — the **ancestral DDPM sampler**.
- $\eta=0$ $\Rightarrow$ $\sigma_t=0$, the noise term vanishes and the update is
  **deterministic**:
  $$x_s=\sqrt{\bar\alpha_s}\,\hat x_0+\sqrt{1-\bar\alpha_s}\,\epsilon_\theta(x_t,t).$$
  This is the Euler discretization of a *probability-flow ODE*; because there is
  no randomness, you can use a coarse subset of timesteps with little loss.

**Acceleration.** Choose a sub-sequence $\{\tau_i\}\subset\{1,\dots,T\}$ and apply
the update along it. With $S$ steps the cost is $S$ network calls instead of $T$.

## 4. Model — locally defined DDPM-style eps-network

In [ ]:
# ===== actual implementation from ddim.py =====
from __future__ import annotations

import numpy as np

SEED = 0

def _ddim_step_numpy(x_t, eps, abar_t, abar_s, eta, rng):
    x0 = (x_t - np.sqrt(1 - abar_t) * eps) / np.sqrt(abar_t)
    sigma = eta * np.sqrt((1 - abar_s) / (1 - abar_t)) * np.sqrt(1 - abar_t / abar_s)
    dir_xt = np.sqrt(np.maximum(1 - abar_s - sigma ** 2, 0.0)) * eps
    noise = sigma * rng.normal(size=x_t.shape) if eta > 0 else 0.0
    return np.sqrt(abar_s) * x0 + dir_xt + noise

import torch

import torch.nn as nn

import torch.nn.functional as F

def get_device() -> torch.device:
    if torch.cuda.is_available():
        return torch.device("cuda")
    if torch.backends.mps.is_available():
        return torch.device("mps")
    return torch.device("cpu")

def sinusoidal_embedding(t: torch.Tensor, dim: int) -> torch.Tensor:
    half = dim // 2
    freqs = torch.exp(-np.log(10000.0) * torch.arange(half, device=t.device) / half)
    args = t.float()[:, None] * freqs[None, :]
    return torch.cat([torch.sin(args), torch.cos(args)], dim=1)

def demo():
    np.random.seed(SEED); torch.manual_seed(SEED)
    torch.set_num_threads(1)  # tiny model: 1 thread avoids CPU thrashing
    from sklearn.datasets import make_moons
    X, _ = make_moons(2000, noise=0.05, random_state=SEED)
    X = ((X - X.mean(0)) / X.std(0)).astype(np.float32)

    m = DDIM(data_dim=2, T=200).fit(X, epochs=400)
    print(f"DDIM (eps-model) final MSE = {m.history[-1]:.4f}")

    for steps, eta, tag in [(200, 0.0, "det 200"), (20, 0.0, "det  20"), (20, 1.0, "stoch 20")]:
        s = m.sample(2000, steps=steps, eta=eta)
        print(f"  {tag}-step: samp mean={s.mean(0).round(2)} std={s.std(0).round(2)}")
    print(f"  data:        mean={X.mean(0).round(2)} std={X.std(0).round(2)}")


class EpsMLP(nn.Module):
    """Noise predictor eps_theta(x_t, t) — identical in spirit to the DDPM net."""

    def __init__(self, data_dim: int = 2, hidden: int = 128, t_dim: int = 32):
        super().__init__()
        self.t_dim = t_dim
        self.net = nn.Sequential(
            nn.Linear(data_dim + t_dim, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, hidden), nn.SiLU(),
            nn.Linear(hidden, data_dim))

    def forward(self, x: torch.Tensor, t: torch.Tensor) -> torch.Tensor:
        return self.net(torch.cat([x, sinusoidal_embedding(t, self.t_dim)], dim=1))


class DDIM(nn.Module):
    r"""
    Trains a standard eps-model (DDPM simplified loss) and samples it with the
    DDIM update over a chosen sub-sequence of timesteps.

    DDIM step from time t to an earlier time s (with x0-prediction):
        x0      = (x_t - sqrt(1-abar_t) eps) / sqrt(abar_t)
        sigma   = eta sqrt((1-abar_s)/(1-abar_t)) sqrt(1 - abar_t/abar_s)
        x_s     = sqrt(abar_s) x0 + sqrt(1-abar_s-sigma^2) eps + sigma z
    eta=0 is deterministic; eta=1 reproduces DDPM ancestral sampling.
    """

    def __init__(self, data_dim: int = 2, T: int = 200, hidden: int = 128):
        super().__init__()
        self.T = T
        self.model = EpsMLP(data_dim, hidden)
        betas = torch.linspace(1e-4, 0.02, T)
        abars = torch.cumprod(1.0 - betas, dim=0)
        self.register_buffer("abars", abars)

    def q_sample(self, x0, t, eps):
        ab = self.abars[t].unsqueeze(1)
        return ab.sqrt() * x0 + (1.0 - ab).sqrt() * eps

    def loss(self, x0):
        n = len(x0)
        t = torch.randint(0, self.T, (n,), device=x0.device)
        eps = torch.randn_like(x0)
        return F.mse_loss(self.model(self.q_sample(x0, t, eps), t), eps)

    def fit(self, X, epochs: int = 400, batch: int = 256, lr: float = 2e-3):
        dev = get_device()
        self.to(dev)
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        opt = torch.optim.Adam(self.parameters(), lr=lr)
        self.history = []
        for _ in range(epochs):
            perm = torch.randperm(len(X), device=dev)
            tot = 0.0
            for s in range(0, len(X), batch):
                loss = self.loss(X[perm[s:s + batch]])
                opt.zero_grad(); loss.backward(); opt.step()
                tot += loss.item()
            self.history.append(tot / max(1, len(X) // batch))
        return self

    @torch.no_grad()
    def sample(self, n: int, steps: int = 20, eta: float = 0.0, data_dim: int = 2):
        """Accelerated DDIM sampling over `steps` of the T timesteps."""
        dev = next(self.parameters()).device
        # evenly spaced sub-sequence of timesteps, descending to 0
        seq = torch.linspace(0, self.T - 1, steps, device=dev).round().long()
        seq = torch.unique(seq).sort(descending=True).values
        x = torch.randn(n, data_dim, device=dev)
        for i, ti in enumerate(seq):
            t = torch.full((n,), int(ti), device=dev, dtype=torch.long)
            eps = self.model(x, t)
            abar_t = self.abars[ti]
            abar_s = self.abars[seq[i + 1]] if i + 1 < len(seq) else torch.tensor(1.0, device=dev)
            x0 = (x - (1 - abar_t).sqrt() * eps) / abar_t.sqrt()
            if i + 1 < len(seq):
                sigma = eta * ((1 - abar_s) / (1 - abar_t)).sqrt() * (1 - abar_t / abar_s).sqrt()
                dir_xt = torch.clamp(1 - abar_s - sigma ** 2, min=0.0).sqrt() * eps
                noise = sigma * torch.randn_like(x) if eta > 0 else 0.0
                x = abar_s.sqrt() * x0 + dir_xt + noise
            else:
                x = x0  # final step lands on the clean prediction
        return x.cpu().numpy()

## 5. Training / sampling — DDPM loss + DDIM accelerated sampler

In [ ]:
# ===== actual implementation from ddim.py =====
class DDIM(nn.Module):
    r"""
    Trains a standard eps-model (DDPM simplified loss) and samples it with the
    DDIM update over a chosen sub-sequence of timesteps.

    DDIM step from time t to an earlier time s (with x0-prediction):
        x0      = (x_t - sqrt(1-abar_t) eps) / sqrt(abar_t)
        sigma   = eta sqrt((1-abar_s)/(1-abar_t)) sqrt(1 - abar_t/abar_s)
        x_s     = sqrt(abar_s) x0 + sqrt(1-abar_s-sigma^2) eps + sigma z
    eta=0 is deterministic; eta=1 reproduces DDPM ancestral sampling.
    """

    def __init__(self, data_dim: int = 2, T: int = 200, hidden: int = 128):
        super().__init__()
        self.T = T
        self.model = EpsMLP(data_dim, hidden)
        betas = torch.linspace(1e-4, 0.02, T)
        abars = torch.cumprod(1.0 - betas, dim=0)
        self.register_buffer("abars", abars)

    def q_sample(self, x0, t, eps):
        ab = self.abars[t].unsqueeze(1)
        return ab.sqrt() * x0 + (1.0 - ab).sqrt() * eps

    def loss(self, x0):
        n = len(x0)
        t = torch.randint(0, self.T, (n,), device=x0.device)
        eps = torch.randn_like(x0)
        return F.mse_loss(self.model(self.q_sample(x0, t, eps), t), eps)

    def fit(self, X, epochs: int = 400, batch: int = 256, lr: float = 2e-3):
        dev = get_device()
        self.to(dev)
        X = torch.as_tensor(X, dtype=torch.float32, device=dev)
        opt = torch.optim.Adam(self.parameters(), lr=lr)
        self.history = []
        for _ in range(epochs):
            perm = torch.randperm(len(X), device=dev)
            tot = 0.0
            for s in range(0, len(X), batch):
                loss = self.loss(X[perm[s:s + batch]])
                opt.zero_grad(); loss.backward(); opt.step()
                tot += loss.item()
            self.history.append(tot / max(1, len(X) // batch))
        return self

    @torch.no_grad()
    def sample(self, n: int, steps: int = 20, eta: float = 0.0, data_dim: int = 2):
        """Accelerated DDIM sampling over `steps` of the T timesteps."""
        dev = next(self.parameters()).device
        # evenly spaced sub-sequence of timesteps, descending to 0
        seq = torch.linspace(0, self.T - 1, steps, device=dev).round().long()
        seq = torch.unique(seq).sort(descending=True).values
        x = torch.randn(n, data_dim, device=dev)
        for i, ti in enumerate(seq):
            t = torch.full((n,), int(ti), device=dev, dtype=torch.long)
            eps = self.model(x, t)
            abar_t = self.abars[ti]
            abar_s = self.abars[seq[i + 1]] if i + 1 < len(seq) else torch.tensor(1.0, device=dev)
            x0 = (x - (1 - abar_t).sqrt() * eps) / abar_t.sqrt()
            if i + 1 < len(seq):
                sigma = eta * ((1 - abar_s) / (1 - abar_t)).sqrt() * (1 - abar_t / abar_s).sqrt()
                dir_xt = torch.clamp(1 - abar_s - sigma ** 2, min=0.0).sqrt() * eps
                noise = sigma * torch.randn_like(x) if eta > 0 else 0.0
                x = abar_s.sqrt() * x0 + dir_xt + noise
            else:
                x = x0  # final step lands on the clean prediction
        return x.cpu().numpy()

## 6. Train once, then sample with full vs few steps (eta = 0 / 1)

In [ ]:
demo()

## 7. Visualization — sample quality vs number of steps

In [ ]:
import matplotlib; matplotlib.use("Agg")
import numpy as np, matplotlib.pyplot as plt
from sklearn.datasets import make_moons
import ddim as M

X, _ = make_moons(2000, noise=0.05, random_state=0)
X = ((X - X.mean(0)) / X.std(0)).astype("float32")
m = M.DDIM(data_dim=2, T=200).fit(X, epochs=400)

configs = [(200, 0.0, "det, 200 steps"),
           (20, 0.0, "det, 20 steps"),
           (10, 0.0, "det, 10 steps"),
           (20, 1.0, "stochastic, 20 steps")]
fig, axes = plt.subplots(1, 4, figsize=(16, 4))
for ax, (steps, eta, title) in zip(axes, configs):
    s = m.sample(2000, steps=steps, eta=eta)
    ax.scatter(X[:, 0], X[:, 1], s=4, alpha=.2, color="gray")
    ax.scatter(s[:, 0], s[:, 1], s=4, alpha=.4, color="r")
    ax.set_title(title); ax.set_xticks([]); ax.set_yticks([]); ax.set_aspect("equal")
plt.tight_layout(); plt.show()

## 8. Takeaways & pitfalls
- DDIM = **same network, smarter sampler**. No retraining needed.
- $\eta=0$ is deterministic and step-skippable; $\eta=1$ is DDPM.
- Too few steps (e.g. < 10 here) starts to blur fine structure — there is a
  speed/quality frontier.
- The deterministic flow is invertible (encode data $\to$ noise $\to$ data),
  which enables latent interpolation and is the bridge to the **score / SDE
  view** (next file): DDIM is the probability-flow ODE of that SDE.